# Ứng dụng các thuật toán học máy trong chấm điểm tín dụng — Notebook thực hành (HMEQ)

**Đồ án thạc sĩ — Lê Thị Thanh Xuân.** Notebook này chạy lại toàn bộ quy trình của đồ án, từ dữ liệu thô đến mô hình cuối cùng và lớp hỗ trợ quyết định, rồi tự động đối chiếu kết quả với các con số trong bản thảo.

**Phiên bản dùng với repository GitHub.** Mã nguồn (`src/creditscore`), dữ liệu (`data/hmeq.csv`), siêu tham số đã tối ưu và kết quả gốc (`results/v3`) được lấy từ repository. Ô **0.3** tự tìm repository: nếu notebook đang nằm trong repository thì dùng luôn; nếu không, ô này tải repository từ GitHub (repository công khai) hoặc yêu cầu tải lên file ZIP của repository (repository riêng tư).

Kết quả của notebook được ghi vào `results/notebook_run/` và **không ghi đè** kết quả gốc trong `results/v3/` hay mô hình trong `models/`.

### Hướng dẫn từng bước
1. Mở notebook trên Google Colab: <https://colab.research.google.com> → thẻ **GitHub** → dán đường dẫn repository → chọn `notebooks/01_thesis_pipeline.ipynb` (hoặc **File → Upload notebook**). Sửa `REPO_URL` ở ô 0.3 thành đường dẫn repository của bạn.
2. Chọn **Runtime → Run all** (Thời gian chạy → Chạy tất cả).
3. Lần chạy đầu, ô **0.1** cài đúng phiên bản thư viện rồi **tự khởi động lại phiên**; thông báo *“Your session crashed…”* là bình thường. Chọn lại **Runtime → Run all**; lần này ô 0.1 báo đã đúng phiên bản và notebook chạy hết.
4. Đọc kết quả từng bước. Ô cuối (**Bước 13**) tổng hợp mọi giá trị đối chiếu với đồ án (✓ = khớp).

Thời gian chạy với cấu hình mặc định: khoảng **4–6 phút** trên Colab miễn phí.

### Quy trình
| Bước | Nội dung | Mục trong đồ án |
|---|---|---|
| 1 | Nạp dữ liệu (HMEQ / dữ liệu mô phỏng / file của bạn) | 3.1, 3.2.2 |
| 2 | Khám phá dữ liệu: mất cân bằng lớp, dữ liệu khuyết | 3.2.3 |
| 3 | Chia dữ liệu 70/15/15 phân tầng | 3.3 |
| 4 | Tiền xử lý: chỉ báo khuyết, điền khuyết, đặc trưng, WoE/IV | 3.3, 3.4, 4.1 |
| 5 | So sánh kỹ thuật xử lý mất cân bằng lớp | 3.3, 4.3 |
| 6 | Tối ưu siêu tham số và huấn luyện 9 mô hình | 3.6, 3.7, 4.4 |
| 7 | Lựa chọn mô hình ba bước (giải thích được → DeLong → hạng trung bình) | 3.5, 4.5 |
| 8 | Mô hình cuối cùng trên tập kiểm tra | 4.6 |
| 9 | Hiệu chỉnh xác suất vỡ nợ (Platt) | 4.8 |
| 10 | Giải thích bằng SHAP | 4.7 |
| 11 | Nhóm khuyến nghị: Phê duyệt / Xem xét thủ công / Từ chối | 4.9 |
| 12 | 21 kiểm thử mô hình | 3.9, 4.10 |
| 13 | Đối chiếu với đồ án | — |

### Tùy chọn (ô 0.2)
| Biến | Mặc định | Ý nghĩa |
|---|---|---|
| `DATA_SOURCE` | `"hmeq"` | `"hmeq"`: dữ liệu của đồ án (có sẵn); `"synthetic"`: dữ liệu mô phỏng cùng cấu trúc; `"upload"`: file CSV của bạn (cùng 13 cột) |
| `TUNE` | `False` | `False`: dùng siêu tham số đã tối ưu trong đồ án; `True`: chạy lại Bayesian Optimization (50 vòng × 10-fold, **1.5–2 giờ**) |
| `FAST_TUNE` | `False` | Khi `TUNE=True`: chạy thử nhanh (4 vòng × 3-fold, ~5 phút); kết quả **khác** đồ án |
| `RUN_IMBALANCE` | `False` | `True`: chạy lại so sánh 8 kỹ thuật cân bằng lớp (~10–15 phút); `False`: hiển thị kết quả đã lưu |

Chỉ cấu hình mặc định (`"hmeq"`, `TUNE=False`) mới tái lập đúng con số trong đồ án.

## 0.1 Cài đúng phiên bản thư viện
Kết quả trùng khớp đến từng chữ số cần đúng phiên bản scikit-learn, LightGBM, XGBoost, imbalanced-learn. Ô này chỉ cài khi thiếu, sau đó tự khởi động lại phiên.

In [ ]:
import importlib.metadata as md, os, subprocess, sys
PINS = {"numpy": "2.4.4", "pandas": "3.0.2", "scipy": "1.17.1", "scikit-learn": "1.8.0",
        "lightgbm": "4.7.0", "xgboost": "3.2.0", "imbalanced-learn": "0.14.2",
        "scikit-optimize": "0.10.2", "shap": "0.51.0", "statsmodels": None}
def ok(p, v):
    try:
        return v is None or md.version(p) == v
    except md.PackageNotFoundError:
        return False
need = [f"{p}=={v}" if v else p for p, v in PINS.items() if not ok(p, v)]
if need:
    print("Đang cài:", " ".join(need))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
    print("Cài xong — phiên sẽ khởi động lại. Hãy chọn lại Runtime → Run all.")
    os.kill(os.getpid(), 9)
else:
    print("Thư viện đã đúng phiên bản.")

## 0.2 Cấu hình và nạp gói mã nguồn
Sửa các tùy chọn ở đầu ô nếu cần (xem bảng ở đầu notebook).

In [ ]:
DATA_SOURCE = "hmeq"      # "hmeq" | "synthetic" | "upload"
TUNE = False              # True: chạy lại Bayesian Optimization (1.5–2 giờ)
FAST_TUNE = False         # True (khi TUNE=True): chạy thử nhanh, kết quả khác đồ án
RUN_IMBALANCE = False     # True: chạy lại so sánh kỹ thuật cân bằng lớp (~10–15 phút)
SEED = 42

**0.3 Tìm repository.** Nếu chạy trên Colab từ GitHub, sửa `REPO_URL` (repository công khai) hoặc để nguyên và tải lên file ZIP của repository khi được hỏi (repository riêng tư).

In [ ]:
import os, sys, subprocess, zipfile
from pathlib import Path
REPO_URL = "https://github.com/TEN-TAI-KHOAN/Apply-ML-algorithms-in-credit-scoring-modelling.git"   # sửa TEN-TAI-KHOAN thành tên tài khoản GitHub

def find_root(base):
    for p in [base, *base.parents, *sorted(base.glob("*")), *sorted(base.glob("*/*"))]:
        if (p / "src" / "creditscore").is_dir() and (p / "data" / "hmeq.csv").exists():
            return p
    return None

BASE = Path("/content") if Path("/content").exists() else Path.cwd()
ROOT = find_root(Path.cwd()) or find_root(BASE)
if ROOT is None and "TEN-TAI-KHOAN" not in REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, str(BASE / "Apply-ML-algorithms-in-credit-scoring-modelling")], check=True)
    ROOT = find_root(BASE)
if ROOT is None:                      # repository riêng tư: GitHub → Code → Download ZIP, rồi tải file ZIP lên đây
    from google.colab import files
    zp = next(iter(files.upload()))
    zipfile.ZipFile(zp).extractall(BASE)
    ROOT = find_root(BASE)
assert ROOT is not None, "Không tìm thấy repository (cần thư mục src/creditscore và data/hmeq.csv)"
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
print("Repository:", ROOT)

In [ ]:
import ast, json, time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from sklearn.model_selection import StratifiedKFold

from creditscore import protocol_v3 as P, selection as SEL, decision_support as DS
from creditscore.data import RAW_FEATURES, NUM_FEATURES, CAT_FEATURES, TARGET, load_hmeq, split_data, data_summary
from creditscore.features import WoEBinner
from creditscore.synthetic import make_hmeq_like

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
REF = Path("results/v3"); OUT = Path("results/notebook_run"); OUT.mkdir(parents=True, exist_ok=True)
CHECKS = []
def check(step, item, thesis, value, dec=3):
    """Đối chiếu một con số với đồ án (làm tròn như bản thảo)."""
    ok = abs(round(float(value), dec) - float(thesis)) <= 0.5 * 10 ** -dec + 1e-12
    CHECKS.append({"Bước": step, "Nội dung": item, "Đồ án": thesis, "Chạy lại": round(float(value), dec + 1),
                   "Khớp": "✓" if ok else "✗"})
IS_THESIS = DATA_SOURCE == "hmeq" and not TUNE
print("Nạp gói xong. Đối chiếu với đồ án:", "CÓ" if IS_THESIS else "KHÔNG (cấu hình khác đồ án)")

## Bước 1. Nạp dữ liệu (mục 3.1, 3.2.2)
HMEQ gồm 5,960 khoản vay thế chấp bằng giá trị nhà ở; **BAD = 1** là khách hàng vỡ nợ hoặc chậm trả nghiêm trọng (biến mục tiêu quan sát được, không suy ra từ đặc trưng).
12 biến đầu vào: 10 định lượng và 2 định tính (REASON, JOB).

In [ ]:
if DATA_SOURCE == "hmeq":
    df = load_hmeq("data/hmeq.csv")
elif DATA_SOURCE == "synthetic":
    df = make_hmeq_like(5960, seed=SEED)          # dữ liệu mô phỏng: chỉ chứng minh mã chạy được
else:
    from google.colab import files
    up = next(iter(files.upload())); df = load_hmeq(up)
print(df.shape); display(df.head())

## Bước 2. Khám phá dữ liệu (mục 3.2.3)
Hai phát hiện quyết định thiết kế quy trình: (i) **mất cân bằng lớp** 80/20 → không dùng Accuracy, cân bằng lớp trong pipeline;
(ii) **dữ liệu khuyết mang tính thông tin** ở DEBTINC và VALUE → giữ tín hiệu khuyết bằng biến chỉ báo. Notebook `Kiem_tra_Chuong3_HMEQ.ipynb` kiểm tra đầy đủ 209 con số của Chương 3.

In [ ]:
print("Tỷ lệ vỡ nợ: %.2f%% (%d / %d)" % (df[TARGET].mean() * 100, df[TARGET].sum(), len(df)))
print("Accuracy của mô hình dự báo mọi hồ sơ không vỡ nợ: %.2f%%" % ((1 - df[TARGET].mean()) * 100))
miss = pd.DataFrame({"Tỷ lệ khuyết %": df[RAW_FEATURES].isna().mean() * 100,
                     "Khuyết khi BAD=0 %": df.loc[df[TARGET] == 0, RAW_FEATURES].isna().mean() * 100,
                     "Khuyết khi BAD=1 %": df.loc[df[TARGET] == 1, RAW_FEATURES].isna().mean() * 100,
                     "Tỷ lệ vỡ nợ khi khuyết %": [df.loc[df[c].isna(), TARGET].mean() * 100 for c in RAW_FEATURES]})
miss = miss.sort_values("Tỷ lệ khuyết %", ascending=False).round(2); display(miss)
ax = miss[["Khuyết khi BAD=0 %", "Khuyết khi BAD=1 %"]].plot.bar(figsize=(9, 3.5), rot=45, color=["#4C78A8", "#E45756"])
ax.set_ylabel("% hồ sơ"); ax.set_title("Tỷ lệ khuyết theo trạng thái vỡ nợ"); plt.tight_layout(); plt.show()
cc = df.dropna(subset=RAW_FEATURES)
print("Nếu loại hồ sơ khuyết: còn %d hồ sơ, giữ %.1f%% số khoản vỡ nợ, tỷ lệ vỡ nợ giảm còn %.1f%%"
      % (len(cc), cc[TARGET].sum() / df[TARGET].sum() * 100, cc[TARGET].mean() * 100))
if IS_THESIS:
    check("2. Khám phá", "Tỷ lệ vỡ nợ (%)", 19.95, df[TARGET].mean() * 100, 2)
    check("2. Khám phá", "DEBTINC khuyết khi BAD=1 (%)", 66.11, miss.loc["DEBTINC", "Khuyết khi BAD=1 %"], 2)
    check("2. Khám phá", "Loại hồ sơ khuyết: % số vỡ nợ còn lại", 25.2, cc[TARGET].sum() / df[TARGET].sum() * 100, 1)

## Bước 3. Chia dữ liệu (mục 3.3)
Phân tầng theo BAD, `random_state = 42`: **huấn luyện 70%** (tối ưu, huấn luyện), **kiểm định 15%** (lựa chọn mô hình), **kiểm tra 15%** (chỉ đánh giá mô hình cuối cùng, một lần).

In [ ]:
S = split_data(df, SEED); tr, va, te = S["train"], S["valid"], S["test"]
X, y = tr[RAW_FEATURES], tr[TARGET]
display(data_summary(S).round(4))
if IS_THESIS:
    for k, n in [("train", 4172), ("valid", 894), ("test", 894)]:
        check("3. Chia dữ liệu", f"Số hồ sơ {k}", n, len(S[k]), 0)

## Bước 4. Tiền xử lý và xây dựng đặc trưng (mục 3.3, 3.4, 4.1)
Mọi bước được ước lượng **chỉ trên dữ liệu huấn luyện** (của từng fold khi kiểm định chéo):
1. Kiểm định chọn biến chỉ báo khuyết: hồi quy logistic $P(\text{BAD}=1)=\sigma(\beta_0+\beta_1 M_j)$, giữ khi $p<0.05$.
2. Điền khuyết (phương án được Bayesian Optimization chọn cho từng mô hình: chỉ báo + MICE nhóm lịch sử tín dụng; trung vị + chỉ báo; hoặc để nguyên NaN cho XGBoost/LightGBM).
3. YOJ → 5 nhóm thâm niên; đa thức bậc 2 của LOAN, VALUE, CLAGE; One-Hot; StandardScaler.
4. Riêng hồi quy logistic: phân đoạn tối ưu + WoE, giữ biến có IV ≥ 0.02.

In [ ]:
mt = P.missing_indicator_tests(tr); display(mt.round(4))
prep = P.V3Preprocessor().fit(X, y)
print(f"Ma trận đặc trưng: {len(prep.columns_)} cột"); print(prep.columns_)
binner = WoEBinner().fit(X, y); iv = binner.iv_table(); display(iv.round(3))
iv.set_index("feature")["iv"].sort_values().plot.barh(figsize=(6, 4), color="#4C78A8"); plt.axvline(0.02, ls="--", c="grey")
plt.title("Information Value (tập huấn luyện)"); plt.tight_layout(); plt.show()
print("Bảng WoE của DEBTINC:"); display(binner.woe_table("DEBTINC").round(3))
if IS_THESIS:
    check("4. Tiền xử lý", "Số biến chỉ báo khuyết được giữ", 8, mt["keep_indicator"].sum(), 0)
    check("4. Tiền xử lý", "OR chỉ báo khuyết DEBTINC", 17.79, mt.set_index("feature").loc["DEBTINC", "odds_ratio"], 2)
    check("4. Tiền xử lý", "IV DEBTINC", 1.870, iv.set_index("feature").loc["DEBTINC", "iv"], 3)
    check("4. Tiền xử lý", "IV REASON (< 0.02, bị loại)", 0.011, iv.set_index("feature").loc["REASON", "iv"], 3)

## Bước 5. Xử lý mất cân bằng lớp (mục 2.4, 3.3, 4.3)
Tám chiến lược (không xử lý, trọng số lớp, ROS, RUS, SMOTE, ADASYN, SMOTE-Tomek, SMOTE-ENN) được so sánh bằng 5-fold CV trên tập huấn luyện;
việc lấy mẫu chỉ áp dụng trên phần huấn luyện của từng fold. Mặc định hiển thị kết quả đã lưu; đặt `RUN_IMBALANCE = True` để chạy lại.

In [ ]:
if RUN_IMBALANCE:
    t0 = time.time(); imb = P.imbalance_comparison(X, y, P.MODELS, SEED, 5); print(f"{time.time() - t0:.0f}s")
else:
    imb = pd.read_csv(REF / "imbalance.csv"); print("Kết quả đã lưu (reference/v3/imbalance.csv)")
val = "f1" if "f1" in imb.columns else imb.select_dtypes("number").columns[0]
display(imb.pivot_table(index=imb.columns[0], columns=imb.columns[1], values=val).round(3))

## Bước 6. Tối ưu siêu tham số và huấn luyện 9 mô hình (mục 3.6, 3.7, 4.4)
Bayesian Optimization (`BayesSearchCV`, 50 vòng, Stratified 10-fold, tiêu chí F1) tìm đồng thời siêu tham số của mô hình,
**phương án xử lý khuyết** và **kỹ thuật cân bằng lớp** như các siêu tham số dạng phân loại.
Mặc định (`TUNE = False`) dùng siêu tham số tốt nhất đã tìm được trong đồ án; huấn luyện cả 9 mô hình mất khoảng 10 giây.

In [ ]:
tuned = {}
if TUNE:
    n_iter, folds = (4, 3) if FAST_TUNE else (50, 10)
    rows = []
    for m in P.MODELS:
        bp, info, _ = P.tune(m, X, y, n_iter, folds); tuned[m] = bp; rows.append(info)
        print(f"{m:5s} CV F1 = {info['cv_f1']:.4f}  ({info['tuning_seconds']:.0f}s)")
    tuning = pd.DataFrame(rows)
else:
    tuning = pd.read_csv(REF / "tuning.csv")
    tuned = {m: ast.literal_eval(p) for m, p in tuning.set_index("model")["best_params"].items()}
tuning.to_csv(OUT / "tuning.csv", index=False)
display(pd.DataFrame({"Mô hình": [P.LABELS[m] for m in P.MODELS], "Siêu tham số": [tuned[m] for m in P.MODELS]}))

models = {}
for m in P.MODELS:
    models[m], secs = P.fit_with(m, tuned[m], X, y)
ev = pd.DataFrame([{"model": m, **P.metrics(va[TARGET], e.predict(va[RAW_FEATURES]), P.score_of(e, va[RAW_FEATURES]))}
                   for m, e in models.items()]).set_index("model")
print("Kết quả trên tập kiểm định:"); display(ev[["auc", "f1", "precision", "recall", "mcc", "accuracy"]].round(4))
plt.figure(figsize=(6, 5))
for m, e in models.items():
    f, t, _ = roc_curve(va[TARGET], P.score_of(e, va[RAW_FEATURES])); plt.plot(f, t, label=f"{m} ({ev.loc[m, 'auc']:.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=0.8); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC – tập kiểm định"); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Bước 7. Lựa chọn mô hình ba bước (mục 3.5, 4.5)
- **Bước 0 – khả năng giải thích:** chỉ giữ mô hình cho đóng góp chính xác của từng biến (LR, GNB, DT, RF, XGBoost, LightGBM); loại SVM, KNN, MLP.
- **Bước 1 – DeLong:** loại mô hình có AUC thấp hơn có ý nghĩa thống kê (p < 0.05) so với mô hình có AUC cao nhất.
- **Bước 2 – hạng trung bình** trên 5 thước đo (AUC, KS, PR-AUC, Brier sau hiệu chỉnh Platt, MCC) trong nhóm còn lại — không dùng trọng số tự đặt.

Brier cần PD đã hiệu chỉnh, nên mỗi mô hình được hiệu chỉnh Platt trên điểm out-of-fold (10-fold) của tập huấn luyện (~1–2 phút).

In [ ]:
t0 = time.time()
oof = {m: SEL.oof_scores(P.build(m, seed=SEED).set_params(**tuned[m]), X, y.values, P.score_of) for m in models}
print(f"OOF xong sau {time.time() - t0:.0f}s")
sel = SEL.select_champion(va[TARGET].values, {m: P.score_of(e, va[RAW_FEATURES]) for m, e in models.items()},
                          {m: e.predict(va[RAW_FEATURES]) for m, e in models.items()}, oof, y.values)
sel.to_csv(OUT / "selection.csv", index=False)
cols = ["model", "step0_pass", "auc", "auc_lo", "auc_hi", "delta_auc_vs_best", "p_delong", "ks", "pr_auc", "brier", "mcc",
        "cv_auc_std", "step1_pass", "avg_rank_eligible", "selected"]
display(sel[cols].round(4))
champ = sel.query("selected").iloc[0]["model"]
print("Mô hình được chọn:", champ, "-", P.LABELS[champ])
if champ not in ("LGBM", "XGB"):          # lớp giải thích (Bước 10) hỗ trợ LightGBM / XGBoost
    champ = max(["LGBM", "XGB"], key=lambda m: sel.set_index("model").loc[m, "auc"])
    print("Lưu ý: dùng", champ, "cho các bước sau vì lớp giải thích hỗ trợ LightGBM / XGBoost.")
if IS_THESIS:
    T = {"LGBM": (0.974, 0.862, 0.0490, 0.767), "XGB": (0.967, 0.820, 0.0520, 0.776), "RF": (0.962, 0.810, 0.0608, 0.724),
         "SVM": (0.951, 0.836, 0.0729, 0.754), "KNN": (0.935, 0.807, 0.0648, 0.809), "MLP": (0.927, 0.764, 0.0639, 0.768),
         "LR": (0.902, 0.646, 0.0902, 0.565), "GNB": (0.853, 0.602, 0.1011, 0.541), "DT": (0.845, 0.690, 0.1109, 0.655)}
    s = sel.set_index("model")
    for m, (a, k, b, c) in T.items():
        check("7. Lựa chọn", f"{m}: AUC kiểm định", a, s.loc[m, "auc"], 3)
        check("7. Lựa chọn", f"{m}: KS", k, s.loc[m, "ks"], 3)
        check("7. Lựa chọn", f"{m}: Brier (Platt)", b, s.loc[m, "brier"], 4)
        check("7. Lựa chọn", f"{m}: MCC", c, s.loc[m, "mcc"], 3)
    check("7. Lựa chọn", "LightGBM được chọn (1 = đúng)", 1, sel.query("selected").iloc[0]["model"] == "LGBM", 0)

## Bước 8. Mô hình cuối cùng trên tập kiểm tra (mục 4.6)
Mô hình được chọn được huấn luyện lại trên **tập huấn luyện + kiểm định** với siêu tham số đã tối ưu, rồi đánh giá **một lần** trên tập kiểm tra.
Chỉ số chính: AUC (khoảng tin cậy 95% bằng bootstrap 1,000 lần), Gini = 2·AUC − 1, KS; F1, Precision, Recall, MCC ở ngưỡng 0.5.

In [ ]:
trva = pd.concat([tr, va], ignore_index=True)
Xa, ya = trva[RAW_FEATURES], trva[TARGET].values
Xte, yte = te[RAW_FEATURES], te[TARGET].values
final = P.build(champ, seed=SEED).set_params(**tuned[champ]).fit(Xa, ya)
s_te = P.score_of(final, Xte); yhat = final.predict(Xte)
from creditscore.model_testing import bootstrap_auc_ci
mt_ = P.metrics(yte, yhat, s_te); auc = roc_auc_score(yte, s_te)
lo, hi = bootstrap_auc_ci(yte, s_te)                      # bootstrap 1,000 lần, seed 42
f_, t_, _ = roc_curve(yte, s_te); ks = float(np.max(t_ - f_))
lr_te = P.score_of(models["LR"], Xte); diff, p_lr = SEL.delong_test(yte, s_te, lr_te)
res = pd.Series({"AUC": auc, "KTC 95% (dưới)": lo, "KTC 95% (trên)": hi, "Gini": 2 * auc - 1, "KS": ks, "F1": mt_["f1"],
                 "Precision": mt_["precision"], "Recall": mt_["recall"], "MCC": mt_["mcc"], "Số cột đặc trưng": final.named_steps["prep"].transform(Xte).shape[1],
                 "AUC hồi quy logistic (WoE)": roc_auc_score(yte, lr_te), "DeLong p (so với LR)": p_lr})
display(res.round(4).to_frame(P.LABELS[champ]))
pd.DataFrame([{"model": champ, "trained_on": "train+valid", **mt_}]).to_csv(OUT / "final_test.csv", index=False)
if IS_THESIS:
    for k, v, d in [("AUC", 0.982, 3), ("KTC 95% (dưới)", 0.974, 3), ("KTC 95% (trên)", 0.988, 3), ("Gini", 0.964, 3), ("KS", 0.867, 3),
                    ("F1", 0.845, 3), ("Precision", 0.865, 3), ("Recall", 0.826, 3), ("MCC", 0.808, 3), ("AUC hồi quy logistic (WoE)", 0.895, 3), ("Số cột đặc trưng", 32, 0)]:
        check("8. Mô hình cuối cùng", k, v, res[k], d)

## Bước 9. Hiệu chỉnh xác suất vỡ nợ bằng Platt (mục 4.8)
Do cân bằng lớp khi huấn luyện, điểm số chưa phải PD. Hiệu chỉnh Platt: $\text{PD} = \sigma(a\cdot f(\mathbf x) + b)$, với $f(\mathbf x)$ là log-odds của mô hình,
$a, b$ ước lượng trên điểm **out-of-fold 10-fold của tập huấn luyện + kiểm định** (không dùng tập kiểm tra).

In [ ]:
oof_m = np.zeros(len(ya))
for i_tr, i_va in StratifiedKFold(10, shuffle=True, random_state=SEED).split(Xa, ya):
    oof_m[i_va] = DS._margin(clone(final).fit(Xa.iloc[i_tr], ya[i_tr]), Xa.iloc[i_va])
A, B = DS.fit_platt(oof_m, ya); ex = DS.Explainer(final, A, B)
p_raw, p_cal = s_te, ex.pd(Xte)
cal = pd.DataFrame([{"version": "Chưa hiệu chỉnh (sau tái cân bằng)", "brier": brier_score_loss(yte, p_raw), "mean_pd": p_raw.mean(), "observed_rate": yte.mean()},
                    {"version": "Hiệu chỉnh Platt", "brier": brier_score_loss(yte, p_cal), "mean_pd": p_cal.mean(), "observed_rate": yte.mean()}])
cal["platt_a"], cal["platt_b"] = A, B; cal.to_csv(OUT / "calibration_test.csv", index=False)
print(f"a = {A:.4f}, b = {B:.4f}"); display(cal.round(4))
q = pd.qcut(p_cal, 10, labels=False, duplicates="drop")
dec = pd.DataFrame({"PD dự báo": p_cal, "Vỡ nợ thực tế": yte}).groupby(q).mean(); display(dec.round(3))
plt.figure(figsize=(4.5, 4.5)); plt.plot(dec["PD dự báo"], dec["Vỡ nợ thực tế"], "o-", label="Sau hiệu chỉnh")
plt.plot([0, 1], [0, 1], "k--", lw=0.8); plt.xlabel("PD dự báo trung bình"); plt.ylabel("Tỷ lệ vỡ nợ thực tế")
plt.title("Đường hiệu chỉnh – 10 nhóm"); plt.tight_layout(); plt.show()
if IS_THESIS:
    check("9. Hiệu chỉnh", "Brier trước hiệu chỉnh", 0.0521, cal.loc[0, "brier"], 4)
    check("9. Hiệu chỉnh", "Brier sau hiệu chỉnh Platt", 0.0431, cal.loc[1, "brier"], 4)
    check("9. Hiệu chỉnh", "PD trung bình (%)", 21.1, p_cal.mean() * 100, 1)

## Bước 10. Giải thích bằng SHAP (mục 2.1.9, 4.7)
TreeSHAP cho đóng góp **chính xác** của từng biến vào log-odds (sau hiệu chỉnh) của từng hồ sơ; đóng góp của các cột kỹ thuật
(chỉ báo khuyết, đa thức, nhóm YOJ, biến giả) được cộng về 12 biến gốc. Tính cộng tính: $\phi_0 + \sum_j \phi_j = \text{logit}(\text{PD})$.

In [ ]:
phi, base = ex.explain(Xte)
lo_ = np.log(p_cal / (1 - p_cal))
print("Kiểm tra tính cộng tính (sai lệch lớn nhất):", float(np.abs(base + phi.sum(axis=1).values - lo_).max()))
imp = phi.abs().mean().sort_values(ascending=False); share = imp / imp.sum()
display(pd.DataFrame({"mean |SHAP|": imp, "Tỷ trọng %": share * 100}).round(3))
share.sort_values().plot.barh(figsize=(6, 4), color="#E45756"); plt.title("Tỷ trọng |SHAP| trung bình – tập kiểm tra")
plt.tight_layout(); plt.show()
i = int(np.argmax(np.where(yte == 1, p_cal, -1)))          # hồ sơ vỡ nợ có PD cao nhất
rep = ex.report(te.iloc[[i]][RAW_FEATURES])
print(f"Ví dụ hồ sơ #{i}: PD = {rep['pd']:.1%}, nhóm: {rep['band']}")
print("Tăng rủi ro:", rep["risk_up"].round(3).to_dict()); print("Giảm rủi ro:", rep["risk_down"].round(3).to_dict())
if IS_THESIS:
    check("10. SHAP", "DEBTINC: tỷ trọng |SHAP| (%)", 23.8, share["DEBTINC"] * 100, 1)
    check("10. SHAP", "DELINQ: tỷ trọng |SHAP| (%)", 14.8, share["DELINQ"] * 100, 1)

## Bước 11. Nhóm khuyến nghị (mục 4.9)
Trên PD đã hiệu chỉnh: **Phê duyệt** khi PD < 10%; **Từ chối** khi PD ≥ 50%; còn lại **Xem xét thủ công**.
Đây là công cụ hỗ trợ cán bộ tín dụng, không thay thế quyết định cuối cùng.

In [ ]:
band = ex.band(p_cal)
bands = pd.DataFrame({"band": band, "y": yte}).groupby("band").agg(n=("y", "size"), defaults=("y", "sum"), default_rate=("y", "mean"))
bands = bands.reindex(list(DS.BANDS)).reset_index()
bands["share"] = bands["n"] / len(yte); bands["defaults_share"] = bands["defaults"] / yte.sum()
bands.to_csv(OUT / "decision_bands_test.csv", index=False); display(bands.round(4))
ax = bands.set_index("band")["default_rate"].mul(100).plot.bar(rot=0, figsize=(5, 3.5), color=["#54A24B", "#F58518", "#E45756"])
ax.set_ylabel("Tỷ lệ vỡ nợ thực tế (%)"); ax.set_title("Tỷ lệ vỡ nợ theo nhóm khuyến nghị"); plt.tight_layout(); plt.show()
if IS_THESIS:
    b = bands.set_index("band")
    for k, n, r in [("Phê duyệt", 638, 0.8), ("Xem xét thủ công", 74, 31.1), ("Từ chối", 182, 82.4)]:
        check("11. Khuyến nghị", f"{k}: số hồ sơ", n, b.loc[k, "n"], 0)
        check("11. Khuyến nghị", f"{k}: tỷ lệ vỡ nợ (%)", r, b.loc[k, "default_rate"] * 100, 1)
    check("11. Khuyến nghị", "Từ chối chứa % số khoản vỡ nợ", 84.3, b.loc["Từ chối", "defaults_share"] * 100, 1)

## Bước 12. So sánh với mô hình đã lưu và 21 kiểm thử (mục 3.9, 4.10)
Mô hình vừa huấn luyện được lưu vào `results/notebook_run/champion.joblib` và so sánh PD với mô hình trong repository (`models/champion.joblib`, mô hình ứng dụng web dùng).
Sau đó chạy 21 kiểm thử trên mô hình của repository. Kết quả kỳ vọng trong đồ án: **Đạt 19/21** (hai kiểm thử không đạt được thảo luận ở mục 4.10 và trong `reports/deviation_log.md`).

In [ ]:
import joblib
art_nb = {"pipeline": final, "platt_a": A, "platt_b": B, "champion": champ, "params": tuned[champ],
          "feature_names": list(final.named_steps["prep"].columns_)}
joblib.dump(art_nb, OUT / "champion.joblib", compress=3)
pd.DataFrame({"column": art_nb["feature_names"]}).to_csv(OUT / "feature_matrix_final.csv", index=False)
repo_art = joblib.load("models/champion.joblib")
p_repo = DS.Explainer(repo_art["pipeline"], repo_art["platt_a"], repo_art["platt_b"]).pd(Xte)
d = float(np.abs(p_repo - p_cal).max())
print(f"Sai khác PD lớn nhất giữa mô hình vừa huấn luyện và models/champion.joblib: {d:.2e}")
if DATA_SOURCE == "hmeq":
    r = subprocess.run([sys.executable, "scripts/run_model_tests_hmeq.py"], capture_output=True, text=True)
    print(r.stdout[-300:] or r.stderr[-2000:])
    display(Markdown(Path("results/v3/model_tests.md").read_text()))
    tests = pd.read_csv("results/v3/model_tests.csv")
    if IS_THESIS:
        check("12. Kiểm thử", "PD trùng mô hình đã lưu (1 = sai khác < 1e-6)", 1, d < 1e-6, 0)
        check("12. Kiểm thử", "Số kiểm thử đạt (trên 21)", 19, tests["passed"].astype(str).eq("True").sum(), 0)
else:
    print("Bỏ qua: bộ kiểm thử dùng dữ liệu HMEQ.")

## Bước 13. Đối chiếu với đồ án
Mọi giá trị kiểm tra ở các bước trên, so với con số in trong bản thảo (làm tròn như bản thảo). Bảng thứ hai so sánh toàn bộ các bảng kết quả với bản gốc đã dùng trong đồ án (sai khác lớn nhất).

In [ ]:
if not IS_THESIS:
    print("Cấu hình khác đồ án (DATA_SOURCE/TUNE) — không đối chiếu.")
else:
    R = pd.DataFrame(CHECKS)
    display(R.groupby("Bước", sort=False)["Khớp"].agg(Khớp=lambda s: (s == "✓").sum(), Tổng="size"))
    print(f"TỔNG: {(R['Khớp'] == '✓').sum()}/{len(R)} giá trị khớp với đồ án")
    bad = R[R["Khớp"] != "✓"]; display(bad if len(bad) else Markdown("**Không có sai khác.**"))
    rows = []
    for name in ["selection", "final_test", "calibration_test", "decision_bands_test"]:
        a, b = pd.read_csv(REF / f"{name}.csv"), pd.read_csv(OUT / f"{name}.csv")
        num = [c for c in a.select_dtypes("number").columns if c in b.columns and c != "fit_seconds"]
        rows.append({"Bảng": name, "Sai khác lớn nhất": float((a[num] - b[num]).abs().max().max())})
    display(pd.DataFrame(rows))

---
**Ghi chú**
- Biến mục tiêu BAD là kết quả vỡ nợ quan sát được; không có biến đầu vào nào được suy ra từ BAD.
- HMEQ không có ngày tháng, nên đồ án dùng chia ngẫu nhiên phân tầng (không thể kiểm định ngoài thời gian – out-of-time).
- Tập kiểm tra chỉ dùng một lần ở Bước 8–12; mọi bước tiền xử lý, cân bằng lớp, hiệu chỉnh đều ước lượng trên dữ liệu huấn luyện.
- Với `DATA_SOURCE = "synthetic"`, kết quả chỉ chứng minh mã chạy được, không phản ánh chất lượng mô hình.